In [0]:
%sql
CREATE TABLE IF NOT EXISTS bitcoin_hourlyprices_last90days(
  datetime TIMESTAMP,
  prices DOUBLE,
  market_caps DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/bitcoin_hourlyprices_last90days/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bitcoin_5minprices_lastday(
  datetime TIMESTAMP,
  prices DOUBLE,
  market_caps DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/live/bitcoin_5minprices_lastday/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bitcoin_4hourlyohlc_last30days(
  datetime TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/bitcoin_4hourlyohlc_last30days/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bitcoin_30minohlc_lastday(
  datetime TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/live/bitcoin_30minohlc_lastday/'

In [0]:
%sql
MERGE INTO bitcoin_hourlyprices_last90days as target
USING(
  with seperated_val_CTE(
  SELECT from_json(market_caps, 'ARRAY<ARRAY<DOUBLE>>') as m_caps,
  from_json(prices, 'ARRAY<ARRAY<DOUBLE>>') as prc,
  from_json(total_volumes, 'ARRAY<ARRAY<DOUBLE>>') as tot_vol
  from chart_data_table
  ),
  zipped_array(
  SELECT
  explode(arrays_zip(m_caps, prc, tot_vol)) as merged_row
  FROM seperated_val_cte
  )
  select
  FROM_UNIXTIME(merged_row.m_caps[0]/1000) as datetime,
  CAST(merged_row.prc[1] as double) as prices,
  CAST(merged_row.m_caps[1] as double) as market_caps,
  CAST(merged_row.tot_vol[1] as double) as total_volumes
  from zipped_array
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO bitcoin_5minprices_lastday as target
USING(
  with seperated_val_CTE(
  SELECT from_json(market_caps, 'ARRAY<ARRAY<DOUBLE>>') as m_caps,
  from_json(prices, 'ARRAY<ARRAY<DOUBLE>>') as prc,
  from_json(total_volumes, 'ARRAY<ARRAY<DOUBLE>>') as tot_vol
  from live_chart_data_table
  ),
  zipped_array(
  SELECT
  explode(arrays_zip(m_caps, prc, tot_vol)) as merged_row
  FROM seperated_val_cte
  )
  select
  FROM_UNIXTIME(merged_row.m_caps[0]/1000) as datetime,
  CAST(merged_row.prc[1] as double) as prices,
  CAST(merged_row.m_caps[1] as double) as market_caps,
  CAST(merged_row.tot_vol[1] as double) as total_volumes
  from zipped_array
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO bitcoin_4hourlyohlc_last30days as target
USING(
  with exploded_CTE(
  SELECT explode(*) as collec
  FROM ohlc_data_table
)
select
from_unixtime(collec[0]/1000) as datetime,
CAST(collec[1] as DOUBLE) as open,
CAST(collec[2] as double) as high,
CAST(collec[3] as double) as low,
CAST(collec[4] as double) as close
FROM exploded_cte
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
MERGE INTO bitcoin_30minohlc_lastday as target
USING(
  with exploded_CTE(
  SELECT explode(*) as collec
  FROM live_ohlc_data_table
)
select
from_unixtime(collec[0]/1000) as datetime,
CAST(collec[1] as DOUBLE) as open,
CAST(collec[2] as double) as high,
CAST(collec[3] as double) as low,
CAST(collec[4] as double) as close
FROM exploded_cte
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *